# mutability
**Prerequisites:** memory_model, references

**Outcome:** After this notebook you will know exactly which Python objects can be changed after creation, which cannot, why that distinction exists at the memory level, and how it affects every design decision you make with data structures.

## The Problem

In [ ]:
# two containers, one behaves unexpectedly
a = ("api", "worker", "scheduler")
b = ["api", "worker", "scheduler"]

b[0] = "gateway"
print(b)   # works fine

a[0] = "gateway"   # what happens here and why?

## The Concept

A mutable object can have its internal state changed after it is created. An immutable object cannot. The distinction is not about what name points to the object — you can always rebind a name. It is about whether the object itself allows its contents to be modified in place. Lists, dicts, and sets are mutable. Integers, floats, strings, and tuples are immutable. This single property determines whether an object can be used as a dict key, whether passing it to a function is safe, and whether two names sharing it is dangerous.

## Minimal Example

In [ ]:
# mutable — list changes in place, same id
services = ["api", "worker"]
print(id(services))
services.append("scheduler")
print(id(services))   # same id — same object, modified

# immutable — string cannot change, new object created
name = "api"
print(id(name))
name = name + "_v2"
print(id(name))       # different id — new object

## How Python Does It

Immutable objects implement `__hash__` and do not implement `__setitem__` or `__delitem__`. Python's data model uses these dunder methods to control what operations are allowed. When you try to modify an immutable object Python raises a `TypeError` immediately — not because of a runtime check you could bypass, but because the method that would perform the change simply does not exist on that type. Mutability is enforced at the type level, not the value level.

In [ ]:
# strings have no __setitem__ — that is why they are immutable
print(hasattr(str,   "__setitem__"))   # False
print(hasattr(list,  "__setitem__"))   # True
print(hasattr(tuple, "__setitem__"))   # False
print(hasattr(dict,  "__setitem__"))   # True
print(hasattr(set,   "__setitem__"))   # False — sets use add/discard instead

## Building Up

In [ ]:
# mutable types: list, dict, set
jobs = ["job_1", "job_2"]
jobs[0] = "job_0"          # item assignment — allowed
jobs.append("job_3")       # in-place growth — allowed
print(jobs)

In [ ]:
# immutable types: int, float, str, tuple, frozenset, bytes
status = "pending"
try:
    status[0] = "P"         # not allowed
except TypeError as e:
    print(e)

point = (10, 20)
try:
    point[0] = 99           # not allowed
except TypeError as e:
    print(e)

In [ ]:
# immutability enables hashing — mutable objects cannot be dict keys
cache = {}
cache[("us-east", "api")] = "10.0.0.1"    # tuple — hashable — valid key
print(cache)

try:
    cache[["us-east", "api"]] = "10.0.0.1" # list — not hashable — invalid key
except TypeError as e:
    print(e)

In [ ]:
# augmented assignment behaves differently for mutable vs immutable
# immutable: += creates a new object
x = 10
print(id(x))
x += 1
print(id(x))   # different — new int object

# mutable: += modifies in place
items = [1, 2, 3]
print(id(items))
items += [4]
print(id(items))   # same — list extended in place

In [ ]:
# a tuple containing a mutable object is itself immutable
# but the mutable object inside can still be changed
record = (["job_1", "job_2"], "pending")
record[0].append("job_3")   # the list inside the tuple is still mutable
print(record)

try:
    record[1] = "done"       # the tuple slot itself cannot be rebound
except TypeError as e:
    print(e)

In [ ]:
# strings produce new objects on every operation — they never mutate
tag = "pipeline"
print(id(tag))

tag = tag.upper()
print(id(tag))   # different object

tag = tag.replace("PIPELINE", "STREAM")
print(id(tag))   # different object again

## Where It Breaks

In [ ]:
# silent bug: tuple containing a list looks immutable but isn't fully
config = ({"host": "localhost", "port": 5432},)   # tuple with one dict

# this is NOT blocked by the tuple's immutability
config[0]["port"] = 9999
print(config)   # the dict inside changed — the tuple did not protect it

In [ ]:
# building a string in a loop with += is quadratic — each + creates a new string
import time

parts = [str(i) for i in range(10_000)]

start = time.perf_counter()
result = ""
for part in parts:
    result += part   # new string object every iteration
slow = time.perf_counter() - start

start = time.perf_counter()
result = "".join(parts)   # one allocation, one pass
fast = time.perf_counter() - start

print(f"+=    : {slow:.6f}s")
print(f"join(): {fast:.6f}s")

## The Fix

In [ ]:
# use frozenset or tuple for truly immutable containers
permissions = frozenset({"read", "write", "deploy"})

try:
    permissions.add("admin")
except AttributeError as e:
    print(e)   # frozenset has no add method

# frozenset is hashable — can be used as a dict key
policy = {permissions: "standard_user"}
print(policy)

In [ ]:
# use join() instead of += for building strings
log_lines = ["INFO boot", "WARN disk low", "ERROR db timeout"]
report = "\n".join(log_lines)   # single allocation
print(report)

## In a Real System

In [ ]:
# an immutable config snapshot passed safely between pipeline stages
from typing import NamedTuple

class PipelineConfig(NamedTuple):
    host: str
    port: int
    retries: int
    region: str

config = PipelineConfig(host="localhost", port=5432, retries=3, region="us-east")

def extract(cfg):
    print(f"connecting to {cfg.host}:{cfg.port} in {cfg.region}")

def transform(cfg):
    print(f"transforming with {cfg.retries} retries")

extract(config)
transform(config)
# config cannot be accidentally modified by any stage

## Performance

Immutable objects are faster to create in some cases because Python can cache and reuse them (integer interning, string interning). They are also safe to share across threads without locks because no mutation is possible. Mutable objects give you in-place modification which avoids allocation overhead in loops — but that same in-place behaviour is what causes aliasing bugs. Choose immutable types when the data should not change after creation and when you need hashability.

## Anti-Pattern

In [ ]:
# anti-pattern: using a list where a tuple signals intent better

# bad — list implies the coordinates might change
def get_origin():
    return [0, 0]

# good — tuple signals these values are fixed
def get_origin():
    return (0, 0)

# bad — using a list as a dict key attempt
routing_table = {}
try:
    routing_table[["us-east", "api"]] = "10.0.0.1"
except TypeError as e:
    print(e)

# good — tuple is hashable
routing_table[("us-east", "api")] = "10.0.0.1"
print(routing_table)

## Interview Signals

- Why can a tuple be used as a dictionary key but a list cannot?
- Is a tuple containing a list truly immutable? Prove it.
- Why is string concatenation in a loop inefficient and what is the fix?
- What is the difference between rebinding a name and mutating an object?

## Exercise

In [ ]:
def build_report(lines):
    """
    Takes a list of log line strings and returns a single
    newline-separated string report.

    Bug: the current implementation uses string += in a loop.
    Fix it to avoid creating a new string object on every iteration.
    The output must be identical — only the implementation changes.
    """
    report = ""
    for line in lines:
        report += line + "\n"
    return report.rstrip("\n")


logs = ["INFO boot", "WARN disk low", "ERROR db timeout"]
expected = "INFO boot\nWARN disk low\nERROR db timeout"

assert build_report(logs) == expected
assert build_report([]) == ""
assert build_report(["only one"]) == "only one"
print("all assertions passed")